# Modèle climatique : dynamique du cycle du carbone et de la température

Ce cahier implémente un modèle climatique **réduit** inspiré des cadres de type DICE.
Il suit la dynamique du carbone atmosphérique, du carbone océanique, du forçage radiatif et de la température.
L'étape du temps est`Δ` years (annual or 5-year), starting at year `t0`.

---

## State Variables
- **E(t)** : Émissions de CO2 exogènes (GtC par an)
- **M AT(t)** : Masse de carbone dans l'atmosphère (GtC)
- **M UP(t)** : Masse de carbone dans la haute mer (GtC)
- **M LO(t)** : Masse de carbone dans l'océan inférieur (GtC)
- **F(t)** : Radiative forcing (W/m²)  
- **T AT(t)** : anomalie de température atmosphérique (°C)
- **T LO(t)** : anomalie de la température de l'océan inférieur (°C)

---

## Carbon Cycle Equations

$$
M_{AT}(t) = (1 - \Delta b_{12}) M_{AT}(t-1) 
+ \Delta b_{12} \frac{M_{AT0}}{M_{UP0}} M_{UP}(t-1) 
+ \xi E(t-1)
$$

$$
M_{UP}(t) = \Delta b_{12} M_{AT}(t-1) 
+ \left(1 - \Delta b_{12} \tfrac{M_{AT0}}{M_{UP0}} - \Delta b_{23}\right) M_{UP}(t-1) 
+ \Delta b_{23} \frac{M_{UP0}}{M_{LO0}} M_{LO}(t-1)
$$

$$
M_{LO}(t) = \Delta b_{23} M_{UP}(t-1) 
+ \left(1 - \Delta b_{23} \tfrac{M_{UP0}}{M_{LO0}}\right) M_{LO}(t-1)
$$


---

## Radiative Forcing

$$
F(t) = F_{2 \times CO_2} \cdot 
\frac{\ln\!\left(\tfrac{M_{AT}(t-1)}{M_{AT}^{\text{pre}}}\right)}{\ln(2)}
$$

where  
- $F_{2 \times CO_2}$ est le forçage dû au doublement du CO2 (3,36 W/m2),
- $M_{AT}^{\text{pre}}$ is preindustrial atmospheric carbon mass.  

---

## Dynamique de la température

$$
T_{AT}(t) = T_{AT}(t-1) 
+ \Delta c_1 \, F(t) 
- \Delta c_1 \, \frac{F_{2 \times CO_2}}{T_{2 \times CO_2}} \, T_{AT}(t-1) 
- \Delta c_1 c_3 \, \big( T_{AT}(t-1) - T_{LO}(t-1) \big), $$

$$
T_{LO}(t) = T_{LO}(t-1) 
+ \Delta c_4 \, \big( T_{AT}(t-1) - T_{LO}(t-1) \big).
$$

---

## Key Parameters
- $\Delta$: time step (years)  
- $F_{2 \times CO_2}$: forçage pour le doublement du CO2
- $T_{2 \times CO_2}$: sensibilité climatique à l'équilibre (°C par doublement)
- $b_{12}, b_{23}$: carbon transfer rates (atmosphere ↔ ocean)  
- $c_1, c_3, c_4$: heat transfer coefficients  
- $\xi$: facteur de conversion CO2 en C.

---

## Purpose
This notebook will:
1. Initialiser les variables d'état à l'année`t0`.  
2. Simuler la dynamique du carbone et de la température au fil du temps.
3. Explorer l'effet des scénarios d'émissions sur les trajectoires climatiques.

---


In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "requests": "requests",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import climate_models as CM

# A — Dynamique historique du climat

Dans cette section, nous travaillerons avec **données historiques sur les émissions de CO2** recueillies par *Notre monde en données*.
L'ensemble de données fournit ** les émissions annuelles de CO2 par pays** (mesurées en tonnes de CO2) à partir de la révolution industrielle.


In [ ]:
import pandas as pd
import requests

# Fetch the data.
df = pd.read_csv("https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
# OWID short column names are lowercase; restore the labels used below.
df = df.rename(columns={"entity": "Entity", "code": "Code", "year": "Année"})

# Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/annual-co2-emissions-per-country.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()


In [ ]:
# Select World data
df_world = df[df["Entity"] == "World"]

# Or using the code column
df_world = df[df["Code"] == "OWID_WRL"]

# Convert into Tons into GtCO2
world_co2 = df_world["emissions_total"]/(10**9)
year_co2  = df_world["Année"]
print(df_world.head())

### A-1) tracer les trajectoires des émissions de carbone

In [ ]:
# Your code here
pass

### A-2) Nous aimerions simuler la dynamique climatique

#### Initialisation du système climatique

Avant de pouvoir simuler la dynamique climatique, il faut ** mettre le modèle à son équilibre préindustriel**.
Cela signifie que le système était en train de démarrer vers 1750, avant le début des émissions industrielles de CO2.

- **Carbon stocks** are initialized at their **pre-industrial levels**:  
  - $M_{AT0} = M_{AT}^{\text{pre}}$ (atmosphere)  
  - $M_{UP0} = M_{UP}^{\text{pre}}$ (upper ocean)  
  - $M_{LO0} = M_{LO}^{\text{pre}}$ (lower ocean)  

- **Les anomalies température** sont réglées à **zéro** au début:
  - $T_{AT0} = 0$ (no atmospheric anomaly)  
  - $T_{LO0} = 0$ (no ocean anomaly)  

- L'horizon **temps** s'étend de 1749 à 2020 avec des étapes annuelles.

In [ ]:
# create main calibration (default)
p       = CM.Params()
    
# Time definitions
p.Delta = 1         # Annual Data
p.t0    = 1749      #
p.tT    = 2020
p.nT    = (p.tT + p.Delta - p.t0)//p.Delta # update number of periods
    
# Initialize to pre-industrial carbon
p.M_AT0 = p.mat
p.M_UP0 = p.mup
p.M_LO0 = p.mlo

# No Temperatures anomalies
p.T_AT0 = 0
p.T_LO0 = 0

# No methan 
p.M_CH40 = p.mch4

# Initialize the matrix Time x Variables and initialize state variables
path = CM.init_states(p)

# By default: there are no emissions 
# Update the model (with no emissions)
path = CM.update_path(path,p,1750,2020)

# convert into data frame
df = CM.mat_to_df(path,p)


# Show outcome
df

** Exercice :**
1. Quelle est la voie du carbone atmosphérique?`M_AT` et température`T_AT`? Utilisez une parcelle.
2. Quickly comment your outcome.

In [ ]:
# Your code
pass

> ✍️ You written answer here.

### A-3) Alimenter le système climatique avec des émissions de carbone

**Goal**

On charge les émissions historiques de CO2, on les convertit en unités du modèle (GtC/an), on les injecte dans la matrice de simulation`path[:, i_E]`, (iv) mettre à jour le chemin climatique, et (v) tracer et commenter.

---

** Exercice**

1. Remplacer les émissions`path[:, p.i_E]` by `world_co2` alignement par année (déjà fait ci-dessus) et exécution:
   ```python
   CM.update_path(path, p, 1750, 2020)
2. Tracer le résultat en terme d'accumulation de carbone et de température atmosphérique globale.


In [ ]:
# Your code here.
year_co2       = year_co2.reset_index(drop=True)
world_co2      = world_co2.reset_index(drop=True)
tt             = np.where(year_co2 == 2020)[0][0]
path_pollution = path.copy()
pass

### A-4) Emplacement et commentaire: anomalies de température globale

**Goal**

Comparer les anomalies de température globale près de la surface observées avec l'anomalie de température atmosphérique simulée par le modèle obtenue à partir des émissions historiques. Discuter des similitudes et des différences en termes de niveau, de tendance et de variabilité.

---

**Instructions**

1. **Traitement linéaire (comparaison).**
   Emplacement`world_temp` against `world_temp_yr` (série observée) avec`path_pollution[:, p.i_T_AT]` (séries simulées pour les émissions historiques).
   Make sure both are aligned by calendar year.  
   Ajouter une légende, un titre et une grille.

2. **Basic diagnostics.**  
   - Commenter dans quelle mesure la trajectoire simulée reproduit la tendance au réchauffement observée.
   - Discuter des différences entre les premières années et les dernières décennies.
   - Notez que la série observée comprend la variabilité à court terme (ENSO, volcans) non capturée par le modèle simple.
   - Réfléchir à la question de savoir si le modèle sous-estime ou surestime l'ampleur du réchauffement.


In [ ]:
import pandas as pd
import requests

# Fetch the data.
temperatures_data = pd.read_csv("https://ourworldindata.org/grapher/temperature-anomaly.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
temperatures_data = temperatures_data.rename(columns={"entity": "Entity", "code": "Code", "year": "Année"})

# Select column World
temperatures_world = temperatures_data[temperatures_data["Code"] == "OWID_WRL"]

# Select year and global temperatures
world_temp    = temperatures_world["near_surface_temperature_anomaly"]
world_temp    = world_temp.reset_index(drop=True)
world_temp_yr = temperatures_world["Année"]
world_temp_yr = world_temp_yr.reset_index(drop=True)

pass
# Continue to code here


> ✍️ You written answer here.

### A-5) Methane cycles

**Goal**

Étendre le modèle climatique pour tenir compte du méthane (CH4), qui a une durée de vie plus courte que le CO2 mais un potentiel de réchauffement plus fort par molécule. La dynamique du méthane peut être représentée par une simple équation de désintégration et sa contribution au forçage radiatif.

---

** Cycle du méthane (modèle à boîte unique)**

Le méthane s'accumule à travers les émissions et les décompositions avec une durée de vie caractéristique d'environ 12 ans:

$$
M_{CH4}(t) = (1 - \delta \cdot \Delta) \, (M_{CH4}(t-1)  - m_{CH4} ) + \Delta \cdot E_{CH4}(t)
$$

- $M_{CH4}(t)$: atmospheric methane concentration (ppb).  
- $E_{CH4}(t)$: émissions de méthane (en équivalents ppb).
- $\delta \approx 1/12$: annual decay rate.  
- $\Delta$: time step (1 year).
- $m_{CH4}$: niveau préindustriel du méthan.

---

**Forçage radiatif à partir du méthane**

Le forçage du méthane est approximativement proportionnel à la racine carrée de sa concentration:

$$
F_{CH4}(t) = \alpha_{CH4}\,\Big(\sqrt{M_{CH4}(t)} - \sqrt{M_{CH4,0}}\Big)
$$

- $M_{CH4,0} \approx 722$ pb (base préindustrielle).
- $\alpha_{CH4} \approx 0.036 \,\text{W/m}^2 / \sqrt{\text{ppb}}$.  

---

**Total forcing**

Le système climatique répond maintenant au forçage combiné :

$$
F(t) = F_{CO2}(t) + F_{CH4}(t) + F_{\text{ex}}(t)
$$

where $F_{\text{ex}}$ (fixé à zéro) peut capter d'autres gaz à effet de serre ou forçage exogène.

---

**Implementation hints**

- Download Methan data.
- Nourrir le modèle de données méthan.
- Update `update_path` pour simuler à la fois le stock de méthane et son forçage.

---

**Questions**

- Comparer la température globale, le méthan atmosphérique et le carbone dans une sous-plot`1*3`.
- Comment your results.

---


In [ ]:
#Your code here
import pandas as pd
import requests

# Fetch the data.
df_ch4 = pd.read_csv("https://ourworldindata.org/grapher/ghg-emissions-by-gas.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})
df_ch4 = df_ch4.rename(columns={"entity": "Entity", "code": "Code", "year": "Année"})
df_ch4_world = df_ch4[df_ch4["Entity"] == "World"]

print(df_ch4_world)

# Convert ton equivalent tEqCO2 = tCH4*29.8
#                        Mt     = 10^6 tons
#                        ppb    = 2.78*Mt
world_ch4 = df_ch4_world["annual_emissions_ch4_total_co2eq"]/(10**6 * 29.8 * 2.78)
year_ch4  = df_ch4_world["Année"]

# Drop indices
year_ch4       = year_ch4.reset_index(drop=True)
world_ch4      = world_ch4.reset_index(drop=True)
tTmh4          = np.where(year_ch4 == 2020)[0][0]
path_ch4_co2   = path_pollution.copy()

pass

> ✍️ You written answer here.

# B — Forecasting temperatures based on SSP scenarios

**Goal**

Utiliser le GIEC *Pathways socio-économiques partagés (PSS)* en tant qu'avenir alternatif pour les émissions de carbone et examiner comment le système climatique réagit.
Nous commençons par des trajectoires d'émissions qui sont fournies sur une grille **decadale** et les interpolons vers une grille **annuelle** pour être compatibles avec notre modèle climatique.

**Data source**

[Our World in Data – IPCC scenarios explorer](https://ourworldindata.org/explorers/ipcc-scenarios?Metric=Greenhouse+gas+emissions&Sub-metric=Carbon+dioxide+%28CO%E2%82%82%29&Region=Global&country=SSP1+-+Référence~SSP2+-+Référence~SSP3+-+Référence~SSP4+-+Référence~SSP5+-+Référence)

Cet ensemble de données contient des trajectoires d'émissions de CO2 de référence pour les cinq PPU:

- **SSP1 (Sustainability)**  
- **SSP2 (Moyen de la route)**
- **SSP3 (Regional rivalry)**  
- **SSP4 (Inequality)**  
- **SSP5 (Fossil-fueled development)**  

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load
df = pd.read_csv("Notebook_ClimateModels_SSP_data.csv",sep=";",decimal=",")

# Pick year + SSP columns
year = df["Year"]

# Plot
plt.figure
plt.plot(year, df["SSP1"], label="SSP1",color="#1B9E77")
plt.plot(year, df["SSP2"], label="SSP2",color="#D95F02")
plt.plot(year, df["SSP3"], label="SSP3",color="#8C6D31")
plt.plot(year, df["SSP4"], label="SSP4",color="#2CA25F")
plt.plot(year, df["SSP5"], label="SSP5",color="#6F42C1")
plt.grid(True, alpha=0.3)
plt.legend(title="Scenario", ncol=2, frameon=False)
plt.title("Émissions de CO₂ par habitant — références SSP mondiales")
plt.xlabel("Année"); plt.ylabel("CO₂ per capita")
plt.tight_layout();
plt.show()

### B-1) Simuler les trajectoires de température selon les scénarios SSP

**Goal**

Utiliser les émissions interpolées de SSP de **2020 à 2100** comme intrants au modèle climatique, simuler l'anomalie de température atmosphérique (`T_AT`) et comparer les trajectoires de réchauffement qui en résultent.

---

**Instructions**

1. Initialiser le modèle à l'année 2020 en utilisant le dernier état de la simulation historique.
2. Pour chaque scénario SSP, remplissez la colonne d'émissions (`E`) dans la matrice de simulation de 2020 à 2100 avec les valeurs annuelles interpolées.
3. Exécuter la routine de mise à jour du modèle climatique (`CM.update_path`).  
4. Placer ensemble les anomalies de température ** résultantes** pour tous les SSP.

---

**Questions**
1. Commentez votre trajectoire simulée en termes de températures pour différents SSP.


---


In [ ]:
# Your code here
# create main calibration (default)
p2       = CM.Params()
    
# Time definitions
p2.Delta = 1         # Annual Data
p2.t0    = 1749      #
p2.tT    = 2100
p2.nT    = (p2.tT + p2.Delta - p2.t0)//p.Delta # update number of periods
    
# Initialize to pre-industrial carbon
p2.M_AT0 = p.mat
p2.M_UP0 = p.mup
p2.M_LO0 = p.mlo
# No Temperatures anomalies
p2.T_AT0 = 0
p2.T_LO0 = 0

# Initialize the matrix Time x Variables and initialize state variables
path_SSP = CM.init_states(p2)

print(path_SSP)
# By default: there are no emissions 
# Update the model (with no emissions)
path_SSP = CM.update_path(path_SSP,p2,1750,2020)

path_SSP1 = path_SSP.copy()
path_SSP2 = path_SSP.copy()
path_SSP3 = path_SSP.copy()
path_SSP4 = path_SSP.copy()
path_SSP5 = path_SSP.copy()

pass

> ✍️ You written answer here.